<a href="https://colab.research.google.com/github/fatmasenguler/laplacian-minor-hierarchy/blob/main/FOURTHS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
FOURTHS.py
==========

Fourth-order decoupling index of the Laplacian minor hierarchy,

    chi^(i)_{j,k,l} = R_ijkl / (R_ij R_ik R_il)
                    = det K_S / (R_ij R_ik R_il)

where K_S is the 3 x 3 Gram matrix of path overlaps rooted at the
reference residue i (Eq. 18), and R_ijkl = det L(i,j,k,l) / tau is its
determinant (Eq. 16 / C.21).

This is Eq. 19 of the main text and Eq. C.30 of Appendix C.

Convention (main text Section 2.1.3 and Appendix C.6):

    chi = 0  integrated into a cooperative multi-node unit
    chi = 1  partitioned into independent dynamic channels

Interactive script: calculates chi for two proteins and their difference
(Protein 2 - Protein 1).

Output format (4 columns):
  Residue_ID | chi (Protein 1) | chi (Protein 2) | Diff (P2 - P1)

Dependencies: numpy only
"""

import os
import sys
import warnings
import numpy as np

# ======================================================================
# 1. Structure Parsing
# ======================================================================

WATER = {"HOH", "WAT", "DOD", "H2O"}
IONS = {"NA", "CL", "K", "MG", "CA", "ZN", "SO4", "PO4", "GOL", "EDO", "ACT"}


def read_complex(pdb_path, ligand_resnames=None, protein_chains=None,
                 ligand_chains=None, skip_ions=True, model=1):
    """Read protein CA atoms and ligand heavy atoms from a PDB file."""
    ligand_chains = set(ligand_chains or [])
    p_xyz, p_ids, l_xyz, l_ids = [], [], [], []
    current_model, seen_model_record = 1, False

    with open(pdb_path) as fh:
        for line in fh:
            rec = line[:6].strip()

            if rec == "MODEL":
                seen_model_record = True
                try:
                    current_model = int(line[10:14])
                except ValueError:
                    current_model += 1
                continue
            if rec == "ENDMDL":
                continue
            if rec not in ("ATOM", "HETATM"):
                continue
            if seen_model_record and current_model != model:
                continue
            if line[16] not in (" ", "A"):
                continue

            name = line[12:16].strip()
            resname = line[17:20].strip()
            chain = line[21].strip()
            resseq = line[22:26].strip()
            icode = line[26].strip()
            element = line[76:78].strip().upper()
            # PDB hydrogen names may start with a digit ("1HB", "2HG1").
            bare = name.lstrip("0123456789")
            is_hydrogen = (element in ("H", "D")
                           or (not element and bare[:1] in ("H", "D")))

            xyz = (float(line[30:38]), float(line[38:46]), float(line[46:54]))
            label = f"{chain}:{resseq}{icode}:{resname}"

            if rec == "ATOM" and chain in ligand_chains:
                if is_hydrogen:
                    continue
                l_xyz.append(xyz)
                l_ids.append(f"{label}:{name}")

            elif rec == "ATOM":
                if protein_chains and chain not in protein_chains:
                    continue
                if name == "CA":
                    p_xyz.append(xyz)
                    p_ids.append(label)

            else:
                if is_hydrogen or resname in WATER:
                    continue
                if ligand_resnames is not None:
                    if resname not in ligand_resnames:
                        continue
                elif skip_ions and resname in IONS:
                    continue
                l_xyz.append(xyz)
                l_ids.append(f"{label}:{name}")

    p_arr = np.asarray(p_xyz, float).reshape(-1, 3)
    l_arr = np.asarray(l_xyz, float).reshape(-1, 3)
    if len(p_arr) == 0:
        warnings.warn(f"{pdb_path}: no protein CA atoms selected.")
    return p_arr, p_ids, l_arr, l_ids


def parse_residue_numbers(ids):
    """Convert labels into integer numbers and names."""
    nums, names = [], []
    for s in ids:
        field = s.split(":")[1]
        digits = "".join(c for c in field if c.isdigit() or c == "-")
        nums.append(int(digits))
        names.append(s.split(":")[2])
    nums = np.asarray(nums, int)
    return nums, np.asarray(names)


# ======================================================================
# 2. Laplacian & Schur Reduction
# ======================================================================

def build_laplacian(coords, rc=7.8, d0=1.0):
    """Distance-dependent Laplacian of a point set (Eq. 1)."""
    if rc <= 0:
        raise ValueError("rc must be positive")
    if d0 <= 0:
        raise ValueError("d0 must be positive")

    coords = np.asarray(coords, float)
    if coords.ndim != 2 or coords.shape[1] != 3:
        raise ValueError(f"coords must have shape (n, 3), got {coords.shape}")
    if len(coords) == 0:
        raise ValueError("cannot build a Laplacian from an empty node set")

    diff = coords[:, None, :] - coords[None, :, :]
    d = np.sqrt((diff ** 2).sum(-1))

    off = -np.exp(-d / d0)
    off[d > rc] = 0.0
    np.fill_diagonal(off, 0.0)

    L = off.copy()
    np.fill_diagonal(L, -off.sum(axis=1))
    return L


def schur_reduce(L, n_protein, cond_warn=1e12):
    """Kron reduction: eliminate the trailing (ligand) block of L (Eq. 21)."""
    Hpp = L[:n_protein, :n_protein]
    Hpl = L[:n_protein, n_protein:]
    Hlp = L[n_protein:, :n_protein]
    Hll = L[n_protein:, n_protein:]

    if Hll.shape[0] == 0:
        return Hpp.copy()

    cond = np.linalg.cond(Hll)
    if not np.isfinite(cond) or cond > cond_warn:
        warnings.warn(f"H_ll is ill-conditioned (cond = {cond:.2e}). "
                      f"Falling back to least-squares solve.")
        X = np.linalg.lstsq(Hll, Hlp, rcond=None)[0]
    else:
        X = np.linalg.solve(Hll, Hlp)

    return Hpp - Hpl @ X


def build_effective_laplacian(pdb_path, rc=7.8, d0=1.0, model=1,
                              protein_chains=("A",), ligand_chains=("B",),
                              verbose=False):
    """Effective protein Laplacian L_pp - L_pl L_ll^-1 L_lp (Eq. 21)."""
    p_xyz, p_ids, l_xyz, l_ids = read_complex(
        pdb_path,
        protein_chains=protein_chains,
        ligand_chains=ligand_chains,
        model=model
    )

    n_p, n_l = len(p_xyz), len(l_xyz)
    if verbose:
        print(f"{pdb_path}: {n_p} residues (CA), {n_l} ligand heavy atoms")
        if n_l:
            seen, ligand_residues = set(), []
            for s in l_ids:
                key = ":".join(s.split(":")[:3])
                if key not in seen:
                    seen.add(key)
                    ligand_residues.append(key)
            print(f"  ligand residues recognized: {', '.join(ligand_residues)}")

    if n_l == 0:
        if verbose:
            print("  no ligand found: returning plain protein Laplacian")
        return build_laplacian(p_xyz, rc, d0), p_ids

    coords = np.vstack([p_xyz, l_xyz])
    L_full = build_laplacian(coords, rc, d0)
    L_eff = schur_reduce(L_full, n_p)
    L_eff = (L_eff + L_eff.T) / 2

    return L_eff, p_ids


# ======================================================================
# 3. Effective Distances & Higher-Order Decoupling Index
# ======================================================================

def _require_connected_laplacian(L):
    """Raise if L is not numerically a connected Laplacian."""
    L = np.asarray(L, float)
    if L.ndim != 2 or L.shape[0] != L.shape[1]:
        raise ValueError("L must be a square matrix")
    if L.shape[0] < 2:
        raise ValueError("L must contain at least two nodes")

    eig = np.linalg.eigvalsh((L + L.T) / 2)
    tol = 1e-12 * max(float(abs(eig[-1])), 1.0)
    if eig[1] <= tol:
        raise ValueError(
            "The protein network is disconnected at the chosen rc/d0. "
            "Effective resistance between components is not finite."
        )


def effective_distances(L):
    """Pairwise effective distance R_ij = K_ii + K_jj - 2 K_ij (Eq. 7)."""
    _require_connected_laplacian(L)

    N = L.shape[0]
    J = np.ones((N, N)) / N
    K = np.linalg.solve(L + J, np.eye(N)) - J
    diag = np.diag(K)
    R = diag[:, None] + diag[None, :] - 2 * K
    return np.clip(R, 0.0, None)


def path_overlap(R, i, a, b):
    """Path overlap K^(i)_ab = (R_ia + R_ib - R_ab) / 2 (Eq. 9 / 17)."""
    return 0.5 * (R[i, a] + R[i, b] - R[a, b])


def Gram_matrix(R, subset):
    """
    Gram matrix K_S of Eq. C.5, rooted at the reference residue
    i = subset[0]:  (K_S)_pq = (R_ip + R_iq - R_pq) / 2.
    """
    i, rest = subset[0], list(subset[1:])
    m = len(rest)
    K = np.empty((m, m), float)
    for p in range(m):
        for q in range(m):
            K[p, q] = path_overlap(R, i, rest[p], rest[q])
    return K


def higher_order_chi(R, subset):
    """
    Normalized higher-order decoupling index (Eq. 19 / C.23 / C.30).

        chi_S^(i) = det K_S / prod_{s in rest} R_{i,s}

    chi = 0 is fully coupled (a cooperative multi-node unit),
    chi = 1 is fully decoupled (independent channels). This matches the
    third-order index chi^(i)_{j,k} = 1 - (K^(i)_jk)^2 / (R_ij R_ik).

    subset[0] is the REFERENCE residue i and stays FIXED; the remaining
    entries are the other members of S. For the fourth-order scan of
    Figure 4 pass [i, j, k, x] with the structural triad fixed and x
    running over the sequence -- NOT [x, i, j, k].

    det K_S is permutation-symmetric in S, so only the denominator
    depends on which residue is the reference. Moving the reference
    onto the running residue therefore changes the reported profile.

    When x coincides with one of the fixed members, two rows of K_S are
    identical, det K_S = 0, and chi is exactly 0.
    """
    i, rest = subset[0], list(subset[1:])
    K = Gram_matrix(R, subset)
    R_S = float(np.linalg.det(K))
    denom = float(np.prod([R[i, s] for s in rest]))

    if denom <= 0 or np.isnan(R_S):
        return np.nan

    chi = R_S / denom

    # Gram/Hadamard bounds (Eq. C.28). Clip only float dust.
    if np.isfinite(chi):
        if chi < -1e-8 or chi > 1 + 1e-8:
            warnings.warn(f"chi={chi:.3e} outside [0,1]; clipping.")
        chi = np.clip(chi, 0.0, 1.0)

    return chi


# ======================================================================
# 4. Interactive Input Processing
# ======================================================================

def get_inputs():
    """Prompt user for proteins, cutoff, d0, and chain IDs."""
    print("\n" + "=" * 60)
    print("FOURTH-ORDER DECOUPLING INDEX  chi^(i)_{j,k,l}")
    print("=" * 60)

    proteins = []
    for i in [1, 2]:
        while True:
            pdb_file = input(f"\nEnter PDB filename for protein {i} "
                             f"(e.g., 5HED.pdb): ").strip()
            if not pdb_file:
                print("Please enter a filename.")
                continue
            if not os.path.exists(pdb_file):
                print(f"File '{pdb_file}' not found in current directory.")
                print(f"Current directory: {os.getcwd()}")
                continue
            break
        proteins.append(pdb_file)

    print("\n" + "-" * 60)
    print("CHAIN SELECTION")
    print("-" * 60)
    prot_chains = input("Enter protein chain ID(s) "
                        "(comma-separated, default 'A'): ").strip()
    prot_chains = (tuple(c.strip() for c in prot_chains.split(","))
                   if prot_chains else ("A",))
    lig_chains = input("Enter ligand chain ID(s) "
                       "(comma-separated, default 'B'): ").strip()
    lig_chains = (tuple(c.strip() for c in lig_chains.split(","))
                  if lig_chains else ("B",))

    print("\n" + "-" * 60)
    print("PARAMETERS")
    print("-" * 60)

    while True:
        try:
            rc = float(input("Enter cutoff distance in Å "
                             "(default 7.8): ").strip() or "7.8")
            if rc <= 0:
                print("Cutoff must be positive.")
                continue
            break
        except ValueError:
            print("Please enter a valid number.")

    while True:
        try:
            d0 = float(input("Enter temperature/length scale d0 "
                             "(default 1.0): ").strip() or "1.0")
            if d0 <= 0:
                print("d0 must be positive.")
                continue
            break
        except ValueError:
            print("Please enter a valid number.")

    return proteins[0], proteins[1], prot_chains, lig_chains, rc, d0


def prompt_reference_residues(common_res):
    """Prompt for the fixed triad: i is the anchor, j and k the partners."""
    print("\n" + "-" * 60)
    print("REFERENCE RESIDUES i (anchor), j, and k")
    print("-" * 60)
    print(f"Available common residue range: {min(common_res)} to "
          f"{max(common_res)} (Total: {len(common_res)})")

    ref_res = []
    for label, role in (("i", "reference/anchor"), ("j", "partner"),
                        ("k", "partner")):
        while True:
            try:
                res_num = int(input(f"Enter reference residue "
                                    f"{label} ({role}): ").strip())
            except ValueError:
                print("Please enter a valid integer residue number.")
                continue
            if res_num not in common_res:
                print(f"Residue {res_num} is not present in both proteins.")
                continue
            if res_num in ref_res:
                print(f"Residue {label} must be distinct from the previous "
                      f"reference residues ({ref_res}).")
                continue
            ref_res.append(res_num)
            break

    return ref_res[0], ref_res[1], ref_res[2]


# ======================================================================
# 5. Main Processing
# ======================================================================

def main():
    pdb1, pdb2, prot_chains, lig_chains, rc, d0 = get_inputs()

    try:
        print("\nBuilding Laplacian for Protein 1...")
        L1, ids1 = build_effective_laplacian(pdb1, rc=rc, d0=d0,
                                             protein_chains=prot_chains,
                                             ligand_chains=lig_chains,
                                             verbose=True)

        print("\nBuilding Laplacian for Protein 2...")
        L2, ids2 = build_effective_laplacian(pdb2, rc=rc, d0=d0,
                                             protein_chains=prot_chains,
                                             ligand_chains=lig_chains,
                                             verbose=True)
    except Exception as e:
        print(f"\nError building Laplacians: {e}")
        return

    resnums1, resnames1 = parse_residue_numbers(ids1)
    resnums2, resnames2 = parse_residue_numbers(ids2)

    def _index_map(resnums, tag):
        seen = {}
        for idx, r in enumerate(resnums):
            r = int(r)
            if r in seen:
                raise ValueError(
                    f"{tag}: residue number {r} appears more than once in "
                    f"the selected chains. Residue matching would be "
                    f"ambiguous. Restrict the chain selection or renumber."
                )
            seen[r] = idx
        return seen

    res2idx1 = _index_map(resnums1, "Protein 1")
    res2idx2 = _index_map(resnums2, "Protein 2")

    common_res = sorted(set(res2idx1) & set(res2idx2))

    if not common_res:
        print("\nError: No common residue indices found between the two "
              "proteins.")
        return

    print(f"\nResidue count comparison:")
    print(f"  Protein 1 total residues: {len(resnums1)}")
    print(f"  Protein 2 total residues: {len(resnums2)}")
    print(f"  Common residues count:    {len(common_res)}")

    mismatches = [(r, resnames1[res2idx1[r]], resnames2[res2idx2[r]])
                  for r in common_res
                  if resnames1[res2idx1[r]] != resnames2[res2idx2[r]]]
    if mismatches:
        print(f"  residue-name mismatches at {len(mismatches)} positions "
              f"(expected only at mutated sites):")
        for r, n1, n2 in mismatches[:10]:
            print(f"    {r}: {n1} -> {n2}")
        if len(mismatches) > 10:
            print(f"    ... and {len(mismatches) - 10} more")
        if len(mismatches) > 0.1 * len(common_res):
            warnings.warn("More than 10% of common positions disagree in "
                          "residue name. The two structures are probably "
                          "numbered differently.")

    res_i, res_j, res_k = prompt_reference_residues(common_res)

    print("\nCalculating effective distances...")
    R1 = effective_distances(L1)
    R2 = effective_distances(L2)

    i1, j1, k1 = res2idx1[res_i], res2idx1[res_j], res2idx1[res_k]
    i2, j2, k2 = res2idx2[res_i], res2idx2[res_j], res2idx2[res_k]

    base1 = os.path.splitext(os.path.basename(pdb1))[0]
    base2 = os.path.splitext(os.path.basename(pdb2))[0]
    output_file = f"{base1}_vs_{base2}_chi_i{res_i}_j{res_j}_k{res_k}.txt"

    print(f"\nCalculating chi^({res_i})_({res_j},{res_k},x) "
          f"and writing results to: {output_file}")
    with open(output_file, 'w') as f:
        f.write("# Fourth-order decoupling index, Laplacian minor hierarchy\n")
        f.write("# chi^(i)_(j,k,x) = det K_S / (R_ij R_ik R_ix)   (Eq. 19)\n")
        f.write("# chi = 0 fully coupled (cooperative multi-node unit), "
                "chi = 1 fully decoupled (independent channels).\n")
        f.write(f"# i = {res_i} (reference, FIXED), j = {res_j}, "
                f"k = {res_k}, x = running residue\n")
        f.write("# Anchored at i: the reference does not move with x.\n")
        f.write(f"# Protein 1: {pdb1}\n")
        f.write(f"# Protein 2: {pdb2}\n")
        f.write(f"# Protein chains: {','.join(prot_chains)}   "
                f"Ligand chains: {','.join(lig_chains)}\n")
        f.write(f"# Parameters: rc = {rc} Å, d0 = {d0} Å\n")
        f.write(f"# Difference = Protein 2 - Protein 1\n")
        f.write(f"# x = {res_i} is undefined (R_ii = 0); "
                f"x = {res_j} and x = {res_k} are exactly 0.\n")
        f.write("# -------------------------------------------------------"
                "------------\n")
        f.write(f"# {'Res_x':<8} {'P1_chi':<18} {'P2_chi':<18} "
                f"{'Diff_P2-P1':<18}\n")

        for r in common_res:
            x1 = res2idx1[r]
            x2 = res2idx2[r]

            # Reference residue i first: it stays fixed for every x.
            val1 = higher_order_chi(R1, [i1, j1, k1, x1])
            val2 = higher_order_chi(R2, [i2, j2, k2, x2])

            s1 = f"{val1:18.6f}" if np.isfinite(val1) else f"{'NaN':>18}"
            s2 = f"{val2:18.6f}" if np.isfinite(val2) else f"{'NaN':>18}"

            if not (np.isfinite(val1) and np.isfinite(val2)):
                sd = f"{'NaN':>18}"
            else:
                sd = f"{(val2 - val1):18.6f}"

            f.write(f"  {r:<8d} {s1} {s2} {sd}\n")

    print("\n" + "=" * 60)
    print("CALCULATION COMPLETE")
    print("=" * 60)
    print(f"Results saved to: {output_file}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nProcess canceled by user.")
        sys.exit(0)


FOURTH-ORDER DECOUPLING INDEX  chi^(i)_{j,k,l}

Enter PDB filename for protein 1 (e.g., 5HED.pdb): 5HED.pdb

Enter PDB filename for protein 2 (e.g., 5HED.pdb): 5HEB.pdb

------------------------------------------------------------
CHAIN SELECTION
------------------------------------------------------------
Enter protein chain ID(s) (comma-separated, default 'A'): A
Enter ligand chain ID(s) (comma-separated, default 'B'): B

------------------------------------------------------------
PARAMETERS
------------------------------------------------------------
Enter cutoff distance in Å (default 7.8): 7.8
Enter temperature/length scale d0 (default 1.0): 1.0

Building Laplacian for Protein 1...
5HED.pdb: 118 residues (CA), 68 ligand heavy atoms
  ligand residues recognized: B:2:LYS, B:3:ASN, B:4:TYR, B:5:LYS, B:6:GLN, B:7:PHE, B:8:SER, B:9:VAL

Building Laplacian for Protein 2...
5HEB.pdb: 119 residues (CA), 49 ligand heavy atoms
  ligand residues recognized: B:3:ASN, B:4:TYR, B:5:LYS, B:6:G